In [6]:
import time
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def _read_big_csv(path, usecols, dtype_map, sep=",", chunksize=300_000):
    chunks = []
    for chunk in pd.read_csv(
        path,
        usecols=usecols,
        dtype=dtype_map,
        sep=sep,
        chunksize=chunksize,
        low_memory=False
    ):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


def train_random_forest_one_csv(
    file_path="2900.csv",
    feature_cols=("wave", "intensity", "brain_region"),
    target_col="class",
    sep=",",
    chunksize=300_000,
    test_size=0.10,
    random_state=42,
    n_estimators=80,
    max_depth=18,
    min_samples_leaf=10,
    max_samples=0.30,
    min_frequency=None,
    max_categories=None,
    save_model_path="rf_2900_pipeline.joblib",
    save_importance_csv="feature_importance_2900.csv",
    full_report=False
):
    usecols = list(dict.fromkeys([*feature_cols, target_col]))

    dtype_map = {}
    for col in feature_cols:
        if col in ("wave", "intensity"):
            dtype_map[col] = "float32"
        else:
            dtype_map[col] = "string"
    dtype_map[target_col] = "string"

    t0 = time.perf_counter()

    df = _read_big_csv(
        file_path,
        usecols=usecols,
        dtype_map=dtype_map,
        sep=sep,
        chunksize=chunksize
    )

    df = df.dropna(subset=[target_col]).copy()

    for col in feature_cols:
        if col not in df.columns:
            raise ValueError(f"Нет колонки {col!r}")
    if target_col not in df.columns:
        raise ValueError(f"Нет target-колонки {target_col!r}")

    X = df[list(feature_cols)].copy()
    y = df[target_col].astype("category").copy()

    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    for col in categorical_cols:
        X[col] = X[col].astype("category")

    try:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            min_frequency=min_frequency,
            max_categories=max_categories
        )
    except TypeError:
        kwargs = {
            "handle_unknown": "ignore",
            "sparse": True
        }
        if min_frequency is not None:
            kwargs["min_frequency"] = min_frequency
        if max_categories is not None:
            kwargs["max_categories"] = max_categories
        ohe = OneHotEncoder(**kwargs)

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe)
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        sparse_threshold=1.0
    )

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_samples=max_samples,
        bootstrap=True,
        n_jobs=-1,
        random_state=random_state,
        class_weight="balanced_subsample",
        verbose=1
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", rf)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    t1 = time.perf_counter()
    model.fit(X_train, y_train)
    t2 = time.perf_counter()

    y_pred = model.predict(X_test)
    t3 = time.perf_counter()

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0) if full_report else None

    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    importances = model.named_steps["classifier"].feature_importances_

    feature_importance = (
        pd.DataFrame({
            "feature": feature_names,
            "importance": importances
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    feature_importance.to_csv(save_importance_csv, index=False)

    joblib.dump(model, save_model_path, compress=3)

    return {
        "model": model,
        "accuracy": acc,
        "confusion_matrix": cm,
        "classification_report": report,
        "feature_importance": feature_importance,
        "n_rows_total": len(df),
        "n_features_before_ohe": len(feature_cols),
        "n_features_after_ohe": len(feature_names),
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "read_sec": round(t1 - t0, 2),
        "fit_sec": round(t2 - t1, 2),
        "predict_sec": round(t3 - t2, 2),
        "train_shape": X_train.shape,
        "test_shape": X_test.shape
    }


if __name__ == "__main__":
    result = train_random_forest_one_csv(
        file_path="2900.csv",
        feature_cols=("wave", "intensity", "brain_region"),
        target_col="class",
        chunksize=300_000,
        test_size=0.10,
        n_estimators=80,
        max_depth=18,
        min_samples_leaf=10,
        max_samples=0.30,
        min_frequency=None,
        max_categories=None,
        save_model_path="rf_2900_pipeline.joblib",
        save_importance_csv="feature_importance_2900.csv",
        full_report=True
    )

    print("Accuracy:", result["accuracy"])
    print("Rows total:", result["n_rows_total"])
    print("Features before OHE:", result["n_features_before_ohe"])
    print("Features after OHE:", result["n_features_after_ohe"])
    print("Numeric columns:", result["numeric_cols"])
    print("Categorical columns:", result["categorical_cols"])
    print("Train shape:", result["train_shape"])
    print("Test shape:", result["test_shape"])
    print("Read time (sec):", result["read_sec"])
    print("Fit time (sec):", result["fit_sec"])
    print("Predict time (sec):", result["predict_sec"])
    print("Confusion matrix:\n", result["confusion_matrix"])
    print("Top 20 features:\n", result["feature_importance"].head(20))

    if result["classification_report"] is not None:
        print("\nClassification report:\n", result["classification_report"])


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   17.8s
[Parallel(n_jobs=-1)]: Done  80 out of  80 | elapsed:   37.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done  80 out of  80 | elapsed:    0.0s finished


Accuracy: 0.9563394519819101
Rows total: 939750
Features before OHE: 3
Features after OHE: 5
Numeric columns: ['wave', 'intensity']
Categorical columns: ['brain_region']
Train shape: (845775, 3)
Test shape: (93975, 3)
Read time (sec): 1.41
Fit time (sec): 38.44
Predict time (sec): 0.17
Confusion matrix:
 [[29333  2010     0]
 [ 2093 29197     0]
 [    0     0 31342]]
Top 20 features:
                         feature  importance
0                num__intensity    0.389484
1      cat__brain_region_cortex    0.359217
2  cat__brain_region_cerebellum    0.102239
3    cat__brain_region_striatum    0.099175
4                     num__wave    0.049885

Classification report:
               precision    recall  f1-score   support

     control       0.93      0.94      0.93     31343
        endo       0.94      0.93      0.93     31290
         exo       1.00      1.00      1.00     31342

    accuracy                           0.96     93975
   macro avg       0.96      0.96      0.96     939